# Exploratory Data Analysis

Explore sales performance, customer behavior, profitability, and discount patterns. All sales KPIs use delivered order lines only. The dataset and assumed product margins are synthetic.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data/cleaned_data.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
data = pd.read_csv(PROJECT_ROOT / "data/cleaned_data.csv", parse_dates=["Order Date"])
sales = data.loc[data["Order Status"].eq("Delivered")].copy()
sns.set_theme(style="whitegrid", palette="deep")
sales.head()

## KPI definitions

Average order value is calculated from order-level totals, not from individual product lines.

In [ ]:
order_totals = sales.groupby("Order ID")["Total Sales"].sum()
kpis = pd.Series({
    "Delivered revenue": sales["Total Sales"].sum(),
    "Delivered profit": sales["Profit"].sum(),
    "Delivered orders": sales["Order ID"].nunique(),
    "Delivered customers": sales["Customer ID"].nunique(),
    "Average order value": order_totals.mean(),
    "Profit margin": sales["Profit"].sum() / sales["Total Sales"].sum(),
})
display(kpis.to_frame("Value").style.format("{:.2f}"))

## Monthly and annual performance

Aggregate line revenue by month. Seasonality is intentionally represented in the synthetic generator and should not be interpreted as a real market forecast.

In [ ]:
sales["Month"] = sales["Order Date"].dt.to_period("M").dt.to_timestamp()
monthly = sales.groupby("Month", as_index=False)["Total Sales"].sum()
ax = sns.lineplot(data=monthly, x="Month", y="Total Sales", marker="o", color="#176B87")
ax.set(title="Monthly Delivered Revenue", xlabel="Month", ylabel="Revenue ($)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
annual = monthly.assign(Year=monthly["Month"].dt.year).groupby("Year")["Total Sales"].sum()
display(annual.to_frame("Revenue").style.format("${:,.2f}"))

## Product and category performance

Identify the top 10 products by revenue and compare category revenue with profit.

In [ ]:
top_products = (sales.groupby("Product Name")["Total Sales"].sum()
                .nlargest(10).sort_values())
top_products.plot.barh(figsize=(9, 6), color="#4C78A8", title="Top 10 Products by Revenue")
plt.xlabel("Revenue ($)")
plt.tight_layout()
plt.show()

category = sales.groupby("Product Category").agg(
    Revenue=("Total Sales", "sum"), Profit=("Profit", "sum")
)
category["Profit Margin"] = category["Profit"] / category["Revenue"]
display(category.sort_values("Revenue", ascending=False).style.format(
    {"Revenue": "${:,.2f}", "Profit": "${:,.2f}", "Profit Margin": "{:.1%}"}
))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=sales.groupby("Region", as_index=False)["Total Sales"].sum(),
            x="Total Sales", y="Region", ax=axes[0], color="#64CCC5")
axes[0].set_title("Revenue by Region")
sns.barplot(data=sales.groupby("Payment Method", as_index=False)["Total Sales"].sum(),
            x="Total Sales", y="Payment Method", ax=axes[1], color="#9B59B6")
axes[1].set_title("Revenue by Payment Method")
plt.tight_layout()
plt.show()

## Customer behavior and discount impact

Segments use repeat order frequency and recency relative to the latest delivered date present in the dataset. Discount bands compare observed margins; they do not prove discounts caused the difference.

In [ ]:
customer = sales.groupby("Customer ID").agg(
    Spend=("Total Sales", "sum"), Orders=("Order ID", "nunique"),
    Last_Order=("Order Date", "max")
)
customer["Recency"] = (sales["Order Date"].max() - customer["Last_Order"]).dt.days
customer["Segment"] = np.select(
    [customer["Orders"].ge(5) & customer["Recency"].le(90),
     customer["Orders"].ge(3), customer["Recency"].gt(180)],
    ["Champions", "Loyal", "At Risk"], default="New / Occasional"
)
display(customer.groupby("Segment").agg(
    Customers=("Spend", "size"), Revenue=("Spend", "sum"), Average_Spend=("Spend", "mean")
).sort_values("Revenue", ascending=False).style.format(
    {"Revenue": "${:,.2f}", "Average_Spend": "${:,.2f}"}
))

In [ ]:
sales["Discount Band"] = pd.cut(
    sales["Discount"], [-0.001, 0, 0.10, 0.20, 0.80],
    labels=["0%", "1-10%", "11-20%", "21%+"], include_lowest=True
)
discount = sales.groupby("Discount Band", observed=False).agg(
    Revenue=("Total Sales", "sum"), Profit=("Profit", "sum"), Lines=("Order ID", "size")
)
discount["Profit Margin"] = discount["Profit"] / discount["Revenue"]
display(discount.style.format(
    {"Revenue": "${:,.2f}", "Profit": "${:,.2f}", "Profit Margin": "{:.1%}"}
))
discount["Profit Margin"].plot.bar(color="#E07A5F", title="Profit Margin by Discount Band")
plt.ylabel("Profit margin")
plt.tight_layout()
plt.show()

## Reproducible outputs

The script-generated report contains the calculated KPI snapshot and business findings. Re-run the scripts after changing the input data rather than copying figures into documentation.

In [ ]:
insights_path = PROJECT_ROOT / "reports/business_insights.md"
print(insights_path.read_text(encoding="utf-8") if insights_path.exists()
      else "Run scripts/run_analysis.py to generate the insights report.")